# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/python/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object (not as dict or list)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their IDs using the Croissant schema structure.

All entities are referenced by their `@id` fields.

In [ ]:
# List available record sets and their fields by @id
record_sets = list(dataset.metadata.record_sets)
print("Available Record Sets (@id):")
for rs in record_sets:
    print(f"- {rs['@id']} ({rs.get('name', 'No Name')})")

# For each record set, list fields (columns) and their @id
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} ({rs.get('name', 'No Name')})")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    for field in fields:
        print(f"  Field @id: {field['@id']} -- {field.get('name', 'No Name')} (type: {field.get('dataType', 'Unknown')})")

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis.

Use only record set and field `@id`s from the above overview.

In [ ]:
# Extract data from each record set
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show columns from each DataFrame
for rid in record_set_ids:
    print(f"\nDataFrame columns for Record Set {rid}:")
    print(dataframes[rid].columns.tolist())
    print(dataframes[rid].head())

## 4. Exploratory Data Analysis (EDA)

Process the dataset using common EDA steps: filtering, normalization, grouping.

All fields, columns, and record sets are referenced by their `@id` throughout.

In [ ]:
# Select a record set for EDA
# For demonstration, choose the first record set
eda_record_set_id = record_set_ids[0]

df = dataframes[eda_record_set_id]

# Find numeric fields in the record set
numeric_field_ids = []
rs_fields = record_sets[0].get('field', [])
if not isinstance(rs_fields, list):
    rs_fields = [rs_fields]
for field in rs_fields:
    if field.get('dataType', '').startswith('schema:Float') or field.get('dataType', '').startswith('schema:Integer'):
        numeric_field_ids.append(field['@id'])

if numeric_field_ids:
    numeric_field_id = numeric_field_ids[0]
    print(f"Analyzing numeric field {numeric_field_id}")
else:
    numeric_field_id = df.select_dtypes(include='number').columns[0] if len(df.select_dtypes(include='number').columns) else None

# Apply EDA if numeric field found
if numeric_field_id and numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field, if available
    group_field = None
    for field in rs_fields:
        if field.get('dataType', '').startswith('schema:Text') and field['@id'] in df.columns:
            group_field = field['@id']
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field available for EDA in the selected record set.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

Reference fields and columns using their `@id`.

In [ ]:
# Basic visualization: histogram for the numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,6))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If group field exists, plot by group
    if group_field:
        plt.figure(figsize=(8,6))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

In this notebook, we used `mlcroissant` to load and analyze the FAIR^2 dataset: clinical, hormone, imaging, and genetic records for three female patients with non-classical 21-hydroxylase deficiency. Key steps included:
- Metadata and schema loading via Croissant
- Record set and field exploration by `@id`
- Data extraction and basic processing (filtering, normalization, grouping)
- Visualization of numeric and grouped attributes

Limitations of the dataset as noted in documentation include small sample size and restricted generalizability. This workflow demonstrates how FAIR-structured clinical datasets can be programmatically explored for discovery and hypothesis generation.